# Pretrain ijepa on HyperKvasir unlabeled

ViT-S/16 @ 224, global batch 512, 100 epochs. Estimated **~8.8 GPU-hours** (~2 session(s) at the 7.5h guard).

**This notebook is resumable.** It stops cleanly before the session cap, saves full training state (model, EMA target, optimizer, scaler, schedule position) to a Kaggle Dataset, and picks up exactly where it left off next run. Just *Save & Run All* again until it prints `run complete`.

Requires **GPU T4 x2** and Internet ON, plus `KAGGLE_USERNAME`/`KAGGLE_KEY` under Add-ons → Secrets for cross-session checkpointing.

In [ ]:
REPO_URL = "https://github.com/morsalin101/jepa-thesis.git"
BRANCH = "main"
WORKDIR = "/kaggle/working/jepa-thesis"


In [ ]:
import os, subprocess, sys

def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout)
    if r.stderr: print(r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')
    return r

if subprocess.run(f'git ls-remote {REPO_URL}', shell=True,
                  capture_output=True).returncode != 0:
    raise RuntimeError('Cannot reach GitHub — turn Internet ON in the session options.')

if os.path.exists(WORKDIR):
    sh(f'cd {WORKDIR} && git fetch origin && git reset --hard origin/{BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {REPO_URL} {WORKDIR}')
sh(f'cd {WORKDIR} && git log -1 --oneline')
os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)


In [ ]:
sh('pip install -q -r requirements.txt')


In [ ]:
# Report the accelerator and the precision that follows from it.
# T4 (sm_75) has fp16 tensor cores but NO bf16 hardware; P100 (sm_60) has
# neither and runs ~2x slower. The code adapts either way — this cell is
# here so you know what you were given before spending 8 hours on it.
import torch
from src.config import amp_config
print('CUDA devices:', torch.cuda.device_count())
amp = amp_config()
print(amp)
if torch.cuda.device_count() < 2:
    print('\n*** Only one GPU. I-JEPA and MAE will still run correctly, but\n'
          '    SimCLR/MoCo v3 need 2 GPUs to preserve global_batch=512.\n'
          '    Set Session options -> Accelerator -> GPU T4 x2 and re-run. ***')


In [ ]:
from kaggle_secrets import UserSecretsClient
s = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')


In [ ]:
sh('python -m src.engine.pretrain --method ijepa --ckpt-slug morsalin101/jepa-thesis-ckpt --guard-hours 7.5', check=False)


In [ ]:
# If the cell above printed 'N epochs remaining', the session guard stopped it
# cleanly — just Save & Run All again to continue. If it printed 'run complete',
# the exported encoder is in /kaggle/working/weights/ and the next cell ships it.
!ls -la /kaggle/working/weights/ 2>/dev/null || echo 'not finished yet — re-run'


In [ ]:
# Publish the finished encoder (~88 MB) to the shared weights dataset that the
# segmentation notebook mounts. Safe to re-run; a no-op until the run completes.
import glob, json, pathlib, shutil, subprocess

SLUG = 'morsalin101/jepa-thesis-weights'
found = glob.glob('/kaggle/working/weights/*.pt')
if not found:
    print('nothing to publish yet — pretraining has not finished')
else:
    stage = pathlib.Path('/kaggle/working/weights_upload')
    stage.mkdir(exist_ok=True)
    # Carry over any encoders already in the dataset so a new version never
    # drops the other methods' weights.
    for p in glob.glob('/kaggle/input/jepa-thesis-weights/*.pt'):
        shutil.copy(p, stage)
    for p in found:
        shutil.copy(p, stage)
    (stage / 'dataset-metadata.json').write_text(json.dumps(
        {'title': 'jepa-thesis-weights', 'id': SLUG,
         'licenses': [{'name': 'CC0-1.0'}]}, indent=2))
    exists = subprocess.run(['kaggle','datasets','status',SLUG],
                            capture_output=True).returncode == 0
    cmd = (['kaggle','datasets','version','-p',str(stage),'-m',
            'add ijepa','--dir-mode','zip']
           if exists else
           ['kaggle','datasets','create','-p',str(stage),'--dir-mode','zip','--private'])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout or r.stderr)
    print('contents:', sorted(p.name for p in stage.glob('*.pt')))
